In [18]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import re
import pandas as pd
from itertools import product

# ------------------------------
# Molar mass from SMILES
# ------------------------------
def molar_mass_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Descriptors.MolWt(mol)

# ------------------------------
# Parse reaction (SMILES + coefficients)
# ------------------------------
def parse_reaction_smiles(reaction_str):
    left, right = reaction_str.split("->")
    def parse_side(side):
        species = []
        for part in side.split("+"):
            part = part.strip()
            match = re.match(r"(\d*)\s*(\S+)", part)
            coeff = int(match.group(1)) if match.group(1) else 1
            smiles = match.group(2)
            species.append((coeff, smiles))
        return species
    reactants = parse_side(left)
    products = parse_side(right)
    return reactants, products

# ------------------------------
# Compute table for catalyst/ligand combinations with 1 eq for other reactants
# ------------------------------
def compute_masses_table_1eq(reaction, key_reagent_smiles, key_reagent_mass_mg,
                             catalyst_smiles=None, ligand_smiles=None, base_smiles=None,
                             catalyst_eq=0.05, ligand_eq=0.05, base_eq=2.0):
    
    reactants, products = parse_reaction_smiles(reaction)
    
    # --- Find key reagent coefficient ---
    for coeff, smiles in reactants:
        if smiles == key_reagent_smiles:
            key_reagent_coeff = coeff
            break
    else:
        raise ValueError("Key reagent SMILES not found in reactants")
    
    # --- Compute moles of key reagent ---
    MM_key = molar_mass_from_smiles(key_reagent_smiles)
    moles_key = key_reagent_mass_mg / 1000 / MM_key
    
    rows = []
    
    # --- Reactants ---
    for coeff, smiles in reactants:
        MM = molar_mass_from_smiles(smiles)
        if smiles == key_reagent_smiles:
            eq = 1.0  # key reagent
        else:
            eq = 1.1  # other reactants use 1 eq relative to key reagent
        moles_needed = moles_key * eq
        mass_mg = moles_needed * MM * 1000
        rows.append(["Reactant", smiles, round(mass_mg, 5), round(eq, 5)])
    
    # --- Additives ---
    for additive, eq_value, ctype in [(catalyst_smiles, catalyst_eq, "Catalyst"),
                                      (ligand_smiles, ligand_eq, "Ligand"),
                                      (base_smiles, base_eq, "Base")]:
        if additive:
            MM = molar_mass_from_smiles(additive)
            moles_needed = moles_key * eq_value
            mass_mg = moles_needed * MM * 1000
            rows.append([ctype, additive, round(mass_mg, 5), round(eq_value, 5)])
    
    # --- Products ---
    for coeff, smiles in products:
        MM = molar_mass_from_smiles(smiles)
        moles_needed = moles_key * (coeff / key_reagent_coeff)
        mass_mg = moles_needed * MM * 1000
        eq = coeff / key_reagent_coeff
        rows.append(["Product", smiles, round(mass_mg, 5), round(eq, 5)])
    
    df = pd.DataFrame(rows, columns=["Type", "SMILES", "Mass (mg)", "Equivalents"])

    # --- Total mass ---
    total_mass_mg = df["Mass (mg)"].sum()
    
    return df, total_mass_mg

# ------------------------------
# Example usage: multiple catalysts & ligands
# ------------------------------
reaction = "1 BrC1=NC=C(Br)S1 + 1 OB(O)C1=CC=C(OC)C=C1 -> 1 BrC1=CN=C(C2=CC=C(OC)C=C2)S1"
key_reagent_smiles = "BrC1=NC=C(Br)S1"
key_reagent_mass_mg = 50
base_smiles = "[O-]P(=O)([O-])[O-].[K+].[K+].[K+]"

# Define multiple catalysts and ligands
catalyst_list = [
    "CC(=O)[O-].CC(=O)[O-].[Pd+2]",
    "C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.[Pd].[Pd]",
    "Cl[Pd]Cl",
    "C1CC=CCCC=C1.C1CC=CCCC=C1.[Ni]",

]
ligand_list = [
    "CC1(C2=C(C(=CC=C2)P(C3=CC=CC=C3)C4=CC=CC=C4)OC5=C1C=CC=C5P(C6=CC=CC=C6)C7=CC=CC=C7)C",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4)C(C)C",
    "COC1=C(C(=CC=C1)OC)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4",
    "[Pd-4]([P+](c1ccccc1)(c1ccccc1)c1ccccc1)([P+](c1ccccc1)(c1ccccc1)c1ccccc1)([P+](c1ccccc1)(c1ccccc1)c1ccccc1)[P+](c1ccccc1)(c1ccccc1)c1ccccc1",
    "C[C@H](CP(c1ccccc1)c2ccccc2)P(c3ccccc3)c4ccccc4",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=C(C=CC(=C2P(C(C)(C)C)C(C)(C)C)OC)OC)C(C)C.CS(=O)(=O)O.C1=CC=C([C-]=C1)C2=CC=CC=C2N.[Pd]",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=C(C=CC(=C2P(C34CC5CC(C3)CC(C5)C4)C67CC8CC(C6)CC(C8)C7)OC)OC)C(C)C",
    #"[Fe].Cl[Pd]Cl.CC(C)(C)P([C]1[CH][CH][CH][CH]1)C(C)(C)C.CC(C)(C)P([C]2[CH][CH][CH][CH]2)C(C)(C)C",
    #"[Fe].ClCCl.Cl[Pd]Cl.[CH]1[CH][CH][C]([CH]1)P(c2ccccc2)c3ccccc3.[CH]4[CH][CH][C]([CH]4)P(c5ccccc5)c6ccccc",

]

# Loop over all combinations
for catalyst_smiles, ligand_smiles in product(catalyst_list, ligand_list):
    df_masses, total_mass_mg = compute_masses_table_1eq(
        reaction, key_reagent_smiles, key_reagent_mass_mg,
        catalyst_smiles, ligand_smiles, base_smiles
    )
    print(f"--- Combination ---\nCatalyst: {catalyst_smiles}\nLigand: {ligand_smiles}\n")
    display(df_masses)

    print(f"Total mass of all compounds: {round(total_mass_mg, 5)} mg\n")

--- Combination ---
Catalyst: CC(=O)[O-].CC(=O)[O-].[Pd+2]
Ligand: CC1(C2=C(C(=CC=C2)P(C3=CC=CC=C3)C4=CC=CC=C4)OC5=C1C=CC=C5P(C6=CC=CC=C6)C7=CC=CC=C7)C



,Type,SMILES,Mass (mg),Equivalents
0,Reactant,BrC1=NC=C(Br)S1,50.00000,1.00
1,Reactant,OB(O)C1=CC=C(OC)C=C1,34.40469,1.10
2,Catalyst,CC(=O)[O-].CC(=O)[O-].[Pd+2],2.31049,0.05
3,Ligand,CC1(C2=C(C(=CC=C2)P(C3=CC=CC=C3)C4=CC=CC=C4)OC...,5.95489,0.05
4,Base,[O-]P(=O)([O-])[O-].[K+].[K+].[K+],87.37913,2.00
5,Product,BrC1=CN=C(C2=CC=C(OC)C=C2)S1,55.60424,1.00


Total mass of all compounds: 235.65344 mg

--- Combination ---
Catalyst: CC(=O)[O-].CC(=O)[O-].[Pd+2]
Ligand: CC(C)C1=CC(=C(C(=C1)C(C)C)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4)C(C)C



,Type,SMILES,Mass (mg),Equivalents
0,Reactant,BrC1=NC=C(Br)S1,50.00000,1.00
1,Reactant,OB(O)C1=CC=C(OC)C=C1,34.40469,1.10
2,Catalyst,CC(=O)[O-].CC(=O)[O-].[Pd+2],2.31049,0.05
3,Ligand,CC(C)C1=CC(=C(C(=C1)C(C)C)C2=CC=CC=C2P(C3CCCCC...,4.90617,0.05
4,Base,[O-]P(=O)([O-])[O-].[K+].[K+].[K+],87.37913,2.00
5,Product,BrC1=CN=C(C2=CC=C(OC)C=C2)S1,55.60424,1.00


Total mass of all compounds: 234.60472 mg

--- Combination ---
Catalyst: CC(=O)[O-].CC(=O)[O-].[Pd+2]
Ligand: COC1=C(C(=CC=C1)OC)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4



...

Total mass of all compounds: 236.2946 mg

--- Combination ---
Catalyst: CC(=O)[O-].CC(=O)[O-].[Pd+2]
Ligand: [Fe].Cl[Pd]Cl.CC(C)(C)P([C]1[CH][CH][CH][CH]1)C(C)(C)C.CC(C)(C)P([C]2[CH][CH][CH][CH]2)C(C)(C)C



,Type,SMILES,Mass (mg),Equivalents
0,Reactant,BrC1=NC=C(Br)S1,50.00000,1.00
1,Reactant,OB(O)C1=CC=C(OC)C=C1,34.40469,1.10
2,Catalyst,CC(=O)[O-].CC(=O)[O-].[Pd+2],2.31049,0.05
3,Ligand,[Fe].Cl[Pd]Cl.CC(C)(C)P([C]1[CH][CH][CH][CH]1)...,6.70744,0.05
4,Base,[O-]P(=O)([O-])[O-].[K+].[K+].[K+],87.37913,2.00
5,Product,BrC1=CN=C(C2=CC=C(OC)C=C2)S1,55.60424,1.00


Total mass of all compounds: 236.40599 mg



[12:58:36] SMILES Parse Error: unclosed ring for input: '[Fe].ClCCl.Cl[Pd]Cl.[CH]1[CH][CH][C]([CH]1)P(c2ccccc2)c3ccccc3.[CH]4[CH][CH][C]([CH]4)P(c5ccccc5)c6ccccc'


ValueError: Invalid SMILES: [Fe].ClCCl.Cl[Pd]Cl.[CH]1[CH][CH][C]([CH]1)P(c2ccccc2)c3ccccc3.[CH]4[CH][CH][C]([CH]4)P(c5ccccc5)c6ccccc

In [19]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import re
import pandas as pd
from itertools import product

# ------------------------------
# Molar mass from SMILES
# ------------------------------
def molar_mass_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Descriptors.MolWt(mol)

# ------------------------------
# Parse reaction (SMILES + coefficients)
# ------------------------------
def parse_reaction_smiles(reaction_str):
    left, right = reaction_str.split("->")
    def parse_side(side):
        species = []
        for part in side.split("+"):
            part = part.strip()
            match = re.match(r"(\d*)\s*(\S+)", part)
            coeff = int(match.group(1)) if match.group(1) else 1
            smiles = match.group(2)
            species.append((coeff, smiles))
        return species
    reactants = parse_side(left)
    products = parse_side(right)
    return reactants, products

# ------------------------------
# Compute masses for one combination
# ------------------------------
def compute_masses_table_1eq(reaction, key_reagent_smiles, key_reagent_mass_mg,
                             catalyst_smiles=None, ligand_smiles=None, base_smiles=None,
                             catalyst_eq=0.05, ligand_eq=0.05, base_eq=2.0):
    
    reactants, products = parse_reaction_smiles(reaction)
    
    # --- Find key reagent coefficient ---
    for coeff, smiles in reactants:
        if smiles == key_reagent_smiles:
            key_reagent_coeff = coeff
            break
    else:
        raise ValueError("Key reagent SMILES not found in reactants")
    
    # --- Compute moles of key reagent ---
    MM_key = molar_mass_from_smiles(key_reagent_smiles)
    moles_key = key_reagent_mass_mg / 1000 / MM_key
    
    rows = []
    
    # Reactants
    for coeff, smiles in reactants:
        MM = molar_mass_from_smiles(smiles)
        eq = 1.0 if smiles == key_reagent_smiles else 1.1  # 1 eq for all reactants
        moles_needed = moles_key * eq
        mass_mg = moles_needed * MM * 1000
        rows.append(["Reactant", smiles, round(mass_mg, 5), round(eq, 5)])
    
    # Additives
    for additive, eq_value, ctype in [(catalyst_smiles, catalyst_eq, "Catalyst"),
                                      (ligand_smiles, ligand_eq, "Ligand"),
                                      (base_smiles, base_eq, "Base")]:
        if additive:
            MM = molar_mass_from_smiles(additive)
            moles_needed = moles_key * eq_value
            mass_mg = moles_needed * MM * 1000
            rows.append([ctype, additive, round(mass_mg, 5), round(eq_value, 5)])
    
    # Products
    for coeff, smiles in products:
        MM = molar_mass_from_smiles(smiles)
        moles_needed = moles_key * (coeff / key_reagent_coeff)
        mass_mg = moles_needed * MM * 1000
        eq = coeff / key_reagent_coeff
        rows.append(["Product", smiles, round(mass_mg, 5), round(eq, 5)])
    
    df = pd.DataFrame(rows, columns=["Type", "SMILES", "Mass (mg)", "Equivalents"])
    
    return df

# ------------------------------
# Example usage
# ------------------------------
reaction = "1 BrC1=NC=C(Br)S1 + 1 OB(O)C1=CC=C(OC)C=C1 -> 1 BrC1=CN=C(C2=CC=C(OC)C=C2)S1"
key_reagent_smiles = "BrC1=NC=C(Br)S1"
key_reagent_mass_mg = 50
base_smiles = "[O-]P(=O)([O-])[O-].[K+].[K+].[K+]"

catalyst_list = [
    "CC(=O)[O-].CC(=O)[O-].[Pd+2]",
    "C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.[Pd].[Pd]",
    "Cl[Pd]Cl",
    "C1CC=CCCC=C1.C1CC=CCCC=C1.[Ni]",
]

ligand_list = [
    "CC1(C2=C(C(=CC=C2)P(C3=CC=CC=C3)C4=CC=CC=C4)OC5=C1C=CC=C5P(C6=CC=CC=C6)C7=CC=CC=C7)C",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4)C(C)C",
    "COC1=C(C(=CC=C1)OC)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4",
    "[Pd-4]([P+](c1ccccc1)(c1ccccc1)c1ccccc1)([P+](c1ccccc1)(c1ccccc1)c1ccccc1)([P+](c1ccccc1)(c1ccccc1)c1ccccc1)[P+](c1ccccc1)(c1ccccc1)c1ccccc1",
    "C[C@H](CP(c1ccccc1)c2ccccc2)P(c3ccccc3)c4ccccc4",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=C(C=CC(=C2P(C(C)(C)C)C(C)(C)C)OC)OC)C(C)C.CS(=O)(=O)O.C1=CC=C([C-]=C1)C2=CC=CC=C2N.[Pd]",
    "CC(C)C1=CC(=C(C(=C1)C(C)C)C2=C(C=CC(=C2P(C34CC5CC(C3)CC(C5)C4)C67CC8CC(C6)CC(C8)C7)OC)OC)C(C)C",
    #"[Fe].Cl[Pd]Cl.CC(C)(C)P([C]1[CH][CH][CH][CH]1)C(C)(C)C.CC(C)(C)P([C]2[CH][CH][CH][CH]2)C(C)(C)C",
    #"[Fe].ClCCl.Cl[Pd]Cl.[CH]1[CH][CH][C]([CH]1)P(c2ccccc2)c3ccccc3.[CH]4[CH][CH][C]([CH]4)P(c5ccccc5)c6ccccc",
]

# Dictionary to accumulate total masses
total_masses = {}

# Loop over all combinations
for catalyst_smiles, ligand_smiles in product(catalyst_list, ligand_list):
    df_masses = compute_masses_table_1eq(
        reaction, key_reagent_smiles, key_reagent_mass_mg,
        catalyst_smiles, ligand_smiles, base_smiles
    )
    
    # Accumulate totals per SMILES
    for _, row in df_masses.iterrows():
        smiles = row["SMILES"]
        mass = row["Mass (mg)"]
        if smiles in total_masses:
            total_masses[smiles] += mass
        else:
            total_masses[smiles] = mass

# Convert totals to DataFrame
summary_rows = []
for smiles, mass in total_masses.items():
    summary_rows.append([smiles, round(mass, 5)])
summary_df = pd.DataFrame(summary_rows, columns=["SMILES", "Total Mass (mg)"])

print("=== Summary of Total Amounts Needed ===")
display(summary_df)

=== Summary of Total Amounts Needed ===


,SMILES,Total Mass (mg)
0,BrC1=NC=C(Br)S1,1400.00000
1,OB(O)C1=CC=C(OC)C=C1,963.33132
2,CC(=O)[O-].CC(=O)[O-].[Pd+2],16.17343
3,CC1(C2=C(C(=CC=C2)P(C3=CC=CC=C3)C4=CC=CC=C4)OC...,23.81956
4,[O-]P(=O)([O-])[O-].[K+].[K+].[K+],2446.61564
5,BrC1=CN=C(C2=CC=C(OC)C=C2)S1,1556.91872
6,CC(C)C1=CC(=C(C(=C1)C(C)C)C2=CC=CC=C2P(C3CCCCC...,19.62468
7,COC1=C(C(=CC=C1)OC)C2=CC=CC=C2P(C3CCCCC3)C4CCCCC4,16.89992
8,[Pd-4]([P+](c1ccccc1)(c1ccccc1)c1ccccc1)([P+](...,47.57012
9,C[C@H](CP(c1ccccc1)c2ccccc2)P(c3ccccc3)c4ccccc4,16.97876


In [14]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import re
import pandas as pd
from itertools import product
from IPython.display import HTML
from rdkit.Chem.Draw import rdMolDraw2D 

# ------------------------------
# Molar mass from SMILES
# ------------------------------
def molar_mass_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Descriptors.MolWt(mol)

# ------------------------------
# Parse reaction (SMILES + coefficients)
# ------------------------------
def parse_reaction_smiles(reaction_str):
    left, right = reaction_str.split("->")
    def parse_side(side):
        species = []
        for part in side.split("+"):
            part = part.strip()
            match = re.match(r"(\d*)\s*(\S+)", part)
            coeff = int(match.group(1)) if match.group(1) else 1
            smiles = match.group(2)
            species.append((coeff, smiles))
        return species
    reactants = parse_side(left)
    products = parse_side(right)
    return reactants, products

# ------------------------------
# Compute masses for one combination
# ------------------------------
def compute_masses_table_1eq(reaction, key_reagent_smiles, key_reagent_mass_mg,
                             catalyst_smiles=None, ligand_smiles=None, base_smiles=None,
                             catalyst_eq=0.05, ligand_eq=0.05, base_eq=2.0):
    
    reactants, products = parse_reaction_smiles(reaction)
    
    # --- Find key reagent coefficient ---
    key_reagent_coeff = 0
    for coeff, smiles in reactants:
        if smiles == key_reagent_smiles:
            key_reagent_coeff = coeff
            break
    else:
        raise ValueError("Key reagent SMILES not found in reactants")
    
    # --- Compute moles of key reagent ---
    MM_key = molar_mass_from_smiles(key_reagent_smiles)
    moles_key = key_reagent_mass_mg / 1000 / MM_key
    
    rows = []
    
    # Reactants
    for coeff, smiles in reactants:
        MM = molar_mass_from_smiles(smiles)
        eq = 1.0 if smiles == key_reagent_smiles else 1.1 
        moles_needed = moles_key * eq
        mass_mg = moles_needed * MM * 1000
        rows.append(["Reactant", smiles, round(mass_mg, 5), round(eq, 5)])
    
    # Additives
    for additive, eq_value, ctype in [(catalyst_smiles, catalyst_eq, "Catalyst"),
                                     (ligand_smiles, ligand_eq, "Ligand"),
                                     (base_smiles, base_eq, "Base")]:
        if additive:
            MM = molar_mass_from_smiles(additive)
            moles_needed = moles_key * eq_value
            mass_mg = moles_needed * MM * 1000
            rows.append([ctype, additive, round(mass_mg, 5), round(eq_value, 5)])
    
    # Products
    for coeff, smiles in products:
        MM = molar_mass_from_smiles(smiles)
        moles_needed = moles_key * (coeff / key_reagent_coeff)
        mass_mg = moles_needed * MM * 1000
        eq = coeff / key_reagent_coeff
        rows.append(["Product", smiles, round(mass_mg, 5), round(eq, 5)])
    
    df = pd.DataFrame(rows, columns=["Type", "SMILES", "Mass (mg)", "Equivalents"])
    
    return df

# ------------------------------
# Reaction definition
# ------------------------------
reaction = "1 BrC1=NC=C(Br)S1 + 1 OB(O)C1=CC=C(OC)C=C1 -> 1 BrC1=CN=C(C2=CC=C(OC)C=C2)S1"
key_reagent_smiles = "BrC1=NC=C(Br)S1"
key_reagent_mass_mg = 50

# ------------------------------
# Bases, Solvents, Catalysts, and Ligands
# ------------------------------
base_list = [
    "[O-]P(=O)([O-])[O-].[K+].[K+].[K+]", 
    "[Na+].[Na+].[O-]C(=O)[O-]",           
    "[K+].[K+].[O-]C(=O)[O-]",             
    "[Cs+].[Cs+].[O-]C(=O)[O-]",           
    "CC(C)(C)[O-].[K+]"                      
]
solvent_list = ["CN(C)C=O", "Cc1ccccc1"]
catalyst_list = [
    "CC(=O)[O-].CC(=O)[O-].[Pd+2]", 
    "C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.C1=CC=C(C=C1)/C=C/C(=O)/C=C/C2=CC=CC=C2.[Pd].[Pd]", 
    "Cl[Pd]Cl", 
    "C1CC=CCCC=C1.C1CC=CCCC=C1.[Ni]", 
]
ligand_list = [
    "CC1(c2c(Oc3c1cccc3P(c4ccccc4)c5ccccc5)c(P(c6ccccc6)c7ccccc7)ccc2)C", 
    "CC(C)C1=CC(C(C)C)=CC(C(C)C)=C1C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2", 
    "COc1cccc(OC)c1-c2ccccc2P(C3CCCCC3)C4CCCCC4", 
    "[Pd].c1ccc(cc1)P(c2ccccc2)c3ccccc3.c4ccc(cc4)P(c5ccccc5)c6ccccc6.c7ccc(cc7)P(c8ccccc8)c9ccccc9.c%10ccc(cc%10)P(c%11ccccc%11)c%12ccccc%12", 
    "C[C@H](CP(c1ccccc1)c2ccccc2)P(c3ccccc3)c4ccccc4", 
    "COC1=CC=C(C(P(C(C)(C)C)C(C)(C)C)=C1C2=C(C=C(C=C2C(C)C)C(C)C)C(C)C)OC.NC3=C(C=CC=C3)C4=C(C=CC=C4)[Pd]OS(C)(=O)=O", 
    "[Fe].ClCCl.Cl[Pd]Cl.[CH]1[CH][CH][C]([CH]1)P(c2ccccc2)c3ccccc3.[CH]4[CH][CH][C]([CH]4)P(c5ccccc5)c6ccccc6",
    "[Fe].Cl[Pd]Cl.CC(C)(C)P([C]1[CH][CH][CH][CH]1)C(C)(C)C.CC(C)(C)P([C]2[CH][CH][CH][CH]2)C(C)(C)C"
    "COC1=CC=C(OC)C(C2=C(C(C)C)C=C(C(C)C)C=C2C(C)C)=C1P([C@]34C[C@H]5C[C@H](C[C@H](C5)C4)C3)[C@@]6(C[C@H](C7)C8)CC7C[C@H]8C6.NC9=CC=CC=C9C%10=CC=CC=C%10[Pd]OS(C)(=O)=O"

    
]

# ------------------------------
# Run combinations
# ------------------------------
total_masses = []

for catalyst_smiles, ligand_smiles, base_smiles, solvent in product(
        catalyst_list, ligand_list, base_list, solvent_list):

    try:
        df_masses = compute_masses_table_1eq(
            reaction, key_reagent_smiles, key_reagent_mass_mg,
            catalyst_smiles, ligand_smiles, base_smiles
        )

        df_masses["Catalyst"] = catalyst_smiles
        df_masses["Ligand"] = ligand_smiles
        df_masses["Base"] = base_smiles
        df_masses["Solvent"] = solvent

        total_masses.append(df_masses)
    except ValueError as e:
        # Note: This error handling is critical for complex SMILES
        print(f"Skipping combination due to SMILES error: {e}")
        continue

# Final combined DataFrame
final_df = pd.concat(total_masses, ignore_index=True)

print("=== All Combinations Completed ===")

# ------------------------------
# Convert SMILES into RDKit SVG (Fixed Drawing & Bigger Pictures)
# ------------------------------
def smiles_to_svg(smiles, mol_size=(250, 250)): 
    """
    Generates an SVG string from SMILES, bypassing Cairo and forcing 2D coordinates.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f'<div style="color:red; font-size:10px;">Invalid: {smiles}</div>'
    
    try:
        # 1. Force 2D coordinate generation
        AllChem.Compute2DCoords(mol)
        
        # 2. Draw the molecule using the SVG Drawer
        drawer = rdMolDraw2D.MolDraw2DSVG(mol_size[0], mol_size[1])
        
        # 3. SET DRAWING OPTIONS FOR SCALE AND APPEARANCE 
        opts = drawer.drawOptions()
        
        # Ensure the molecule scales up to fill the 250x250 area.
        opts.padding = 0.05 
        
        # Appearance settings
        opts.addAtomIndices = False
        opts.addBondIndices = False
        opts.clearBackground = True
        
        drawer.SetDrawOptions(opts)
        
        drawer.DrawMolecule(mol)
        drawer.FinishDrawing()
        
        # Return the SVG string
        return drawer.GetDrawingText()
    except Exception as e:
        # Fallback if any other issue occurs
        return f'<div style="color:red; font-size:10px;">Final Draw Error: {smiles}</div>'

# ----------------------------------------------------
# New Section: Compute Total Mass for Ordering
# ----------------------------------------------------

# Group by SMILES and sum the Mass (mg)
mass_tally_df = final_df.groupby(["SMILES", "Type"]).agg(
    Total_Mass_mg=('Mass (mg)', 'sum')
).reset_index()

# Round the total mass to two decimal places and convert to grams
mass_tally_df['Total Mass (g)'] = (mass_tally_df['Total_Mass_mg'] / 1000).round(2)
mass_tally_df = mass_tally_df.sort_values(by='Type', ascending=False)


# Add image column to the tally table
mass_tally_df["Structure"] = mass_tally_df["SMILES"].apply(smiles_to_svg)

# Format the final table for display
tally_cols = ["Structure", "Type", "Total Mass (g)", "SMILES"]
final_tally_df = mass_tally_df[tally_cols]


# --- Display the results ---

print("=== Summary: Total Mass Required for All Combinations ===")
display(HTML(final_tally_df.to_html(escape=False)))

print("\n--- Detailed Mass Table (First 100 rows) ---")

# Add image column to the detailed table
final_df["Structure"] = final_df["SMILES"].apply(smiles_to_svg)

# Move picture to the front for detailed table
cols = ["Structure", "Type", "SMILES", "Mass (mg)", "Equivalents", "Catalyst", "Ligand", "Base", "Solvent"]
final_df_detailed = final_df[cols]

HTML(final_df_detailed.head(100).to_html(escape=False))

# ----------------------------------------------------
## 📥 Export to Excel (.xlsx)
# ----------------------------------------------------

# Define the dataframes for export (Exclude SVG column)
df_tally_export = mass_tally_df[["Type", "SMILES", "Total Mass (g)"]].rename(
    columns={'Total Mass (g)': 'Total Mass (g)', 'SMILES': 'SMILES'}
)

# Export the detailed table (Exclude SVG column)
df_detailed_export = final_df[[
    "Type", "SMILES", "Mass (mg)", "Equivalents", "Catalyst", "Ligand", "Base", "Solvent"
]]

# Define the filename
excel_filename = "Suzuki_Mass_Calculations.xlsx"

# Use Pandas ExcelWriter to write multiple sheets
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_tally_export.to_excel(writer, sheet_name='Mass Tally (Order List)', index=False)
    df_detailed_export.to_excel(writer, sheet_name='Detailed Mass Calculations', index=False)

print(f"\n🎉 Data exported successfully to **{excel_filename}** with two sheets:")
print("* **Mass Tally (Order List)**: Total mass required for each unique compound.")
print("* **Detailed Mass Calculations**: Mass for each compound in every single experiment.")

=== All Combinations Completed ===
=== Summary: Total Mass Required for All Combinations ===



--- Detailed Mass Table (First 100 rows) ---

🎉 Data exported successfully to **Suzuki_Mass_Calculations.xlsx** with two sheets:
* **Mass Tally (Order List)**: Total mass required for each unique compound.
* **Detailed Mass Calculations**: Mass for each compound in every single experiment.


In [15]:
import requests
import pandas as pd
import re

def is_valid_cas(cas):
    pattern = r'^\d{2,7}-\d{2}-\d$'
    return bool(re.match(pattern, cas))

def cas_to_smiles(cas):
    if not is_valid_cas(cas):
        return "Invalid CAS number format"
    
    # First, get the PubChem CID for the CAS
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/xref/RN/{cas}/cids/JSON"
    try:
        response = requests.get(cid_url)
        response.raise_for_status()
        data = response.json()
        cids = data.get('IdentifierList', {}).get('CID', [])
        if not cids:
            return "CAS number not found"
        cid = cids[0]

        # Now fetch the SMILES using the CID
        smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSMILES/JSON"
        response = requests.get(smiles_url)
        response.raise_for_status()
        data = response.json()
        smiles = data['PropertyTable']['Properties'][0]['IsomericSMILES']
        return smiles

    except requests.exceptions.RequestException as e:
        return f"Error fetching data: {e}"
    except (KeyError, IndexError):
        return "SMILES not available"

if __name__ == "__main__":
    cas_numbers = [
        "161265-03-8",
        "564483-18-7",
        "657408-07-6",
        "14221-01-3",
        "67884-33-7",
        "1536473-72-9",
        "95464-05-4",
        "95408-45-0",
        "1445972-29-1",
        "3375-31-3",
    ]
    
    molecules = [cas_to_smiles(cas) for cas in cas_numbers] 
    df = pd.DataFrame({'CAS': cas_numbers, 'SMILES': molecules})
    
    output_file = 'cas_to_smiles_results.csv'
    df.to_csv(output_file, index=False)
    
    print("\nResults:")
    print(df)


Results:
            CAS                SMILES
0   161265-03-8  SMILES not available
1   564483-18-7  SMILES not available
2   657408-07-6  SMILES not available
3    14221-01-3  SMILES not available
4    67884-33-7  SMILES not available
5  1536473-72-9  SMILES not available
6    95464-05-4  SMILES not available
7    95408-45-0  SMILES not available
8  1445972-29-1  SMILES not available
9     3375-31-3  SMILES not available
